# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulah-naeem/FlyRank-ml-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method:** Random Forest Classifier (`max_depth=5`)
**Why:** The assignment is a "which first?" ranking task, which means we need continuous scores (probabilities), not just binary labels. A Random Forest provides robust `predict_proba()` outputs that can be used to rank the queue. It natively handles non-linear interactions (e.g., high impressions + low CTR) without needing manual feature crossing like Logistic Regression would. By limiting `max_depth` to 5, we keep the model relatively simple and extract clean feature importances.

In [1]:
# No code needed for method choice, but importing libraries here
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

## 2. Split design

**Design:** `GroupShuffleSplit` on `client_id` (80% Train / 20% Test).
**Why:** Standard train/test splits would randomly mix pages from the same client into both train and test. The model might memorize a specific client's URL structure or domain-specific baseline CTR. By grouping on `client_id`, we guarantee that the model is tested on entirely unseen clients. This is the only honest way to measure how well the model will generalize to a new FlyRank customer.

In [2]:
df = pd.read_csv('../../data/processed/refresh_feature_vector.csv')

# Define features and target
# Exclude target-derived columns (trend_pct, trend_direction) and identifiers to prevent leakage
drop_cols = ['content_id', 'client_id', 'is_declining_label', 'trend_direction', 'trend_pct', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d']
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
features = [c for c in numeric_cols if c not in drop_cols]
target = 'is_declining_label'

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, df[target], groups=df['client_id']))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print(f"Total rows: {len(df)}")
print(f"Train rows: {len(train_df)} | Test rows: {len(test_df)}")
print(f"Train base rate: {train_df[target].mean():.3f} | Test base rate: {test_df[target].mean():.3f}")

Total rows: 30000
Train rows: 23837 | Test rows: 6163
Train base rate: 0.550 | Test base rate: 0.511


## 3. Train + compare vs my baseline

We will train the Random Forest on the train split, then evaluate both the model and the Week 4 baseline rule on the **same held-out test split** using the **same metric** (Precision@50).

In [3]:
# Prepare training data (handle missing values with an indicator to capture content_type missingness signals)
X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

# Train the Random Forest
imputer = SimpleImputer(strategy='median', add_indicator=True)
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)

pipeline = Pipeline([
    ('imputer', imputer),
    ('model', rf)
])

pipeline.fit(X_train, y_train)

# Predict probabilities for the test set
test_df['model_score'] = pipeline.predict_proba(X_test)[:, 1]

# Re-compute Week 4 Baseline score on the EXACT SAME TEST SET
# Rule: (impressions >= 500) * (days_since_last_update >= 180) * (ctr < 1.5) * log1p(impressions)
baseline_mask = (test_df['impressions_90d'] >= 500) & (test_df['days_since_last_update'] >= 180) & (test_df['ctr'] < 1.5)
test_df['baseline_score'] = baseline_mask.astype(int) * np.log1p(test_df['impressions_90d'])

# Evaluate Precision@50
def precision_at_k(df, score_col, target_col, k=50):
    top_k = df.sort_values(by=score_col, ascending=False).head(k)
    return top_k[target_col].mean()

base_rate = test_df[target].mean()
baseline_p50 = precision_at_k(test_df, 'baseline_score', target)
model_p50 = precision_at_k(test_df, 'model_score', target)

# Print Honest Comparison Table
print("========================================")
print(" COMPARISON ON SAME TEST SPLIT")
print("========================================")
print(f"Test Set Base Rate:         {base_rate:.3f}")
print(f"Baseline Precision@50:      {baseline_p50:.3f}")
print(f"Random Forest Precision@50: {model_p50:.3f}")
print("========================================")

 COMPARISON ON SAME TEST SPLIT
Test Set Base Rate:         0.511
Baseline Precision@50:      0.440
Random Forest Precision@50: 0.960


## 4. Errors and interpretation

What features is the model leaning on? Where does it confidently fail? By looking at false positives (where the model scored high, but the page wasn't declining), we can understand its blind spots.

In [4]:
# 1. Feature Importance
# Get feature names after imputation (imputer adds indicator columns for missing values)
feature_names = features + [f"{f}_missing" for i, f in enumerate(features) if i in pipeline.named_steps['imputer'].indicator_.features_]
importances = pipeline.named_steps['model'].feature_importances_

imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=False).head(10)

print("\nTOP 5 FEATURES DRIVING THE MODEL:")
for i, row in imp_df.head(5).iterrows():
    print(f"- {row['feature']}: {row['importance']:.3f}")
    
# 2. Error Analysis (Top 3 Confident False Positives)
# Pages the model ranked very high, but are NOT actually declining
false_positives = test_df[test_df[target] == 0].sort_values(by='model_score', ascending=False).head(3)

print("\nERROR ANALYSIS: TOP 3 CONFIDENT FALSE POSITIVES")
display_cols = ['content_id', 'model_score', 'days_since_last_update', 'impressions_90d', 'ctr', 'content_age_days']
for i, (_, row) in enumerate(false_positives[display_cols].iterrows(), 1):
    print(f"\nFP #{i} (Score: {row['model_score']:.3f})")
    print(f"Content: {row['content_id']}")
    print(f"Days Stale: {row['days_since_last_update']} | Impressions: {row['impressions_90d']} | CTR: {row['ctr']:.2f}% | Age: {row['content_age_days']}")
    print("Interpretation: The model heavily penalizes extreme staleness and low CTR. This page likely fits the typical 'decay' profile perfectly, but happens to be an intentionally evergreen reference page (like a homepage or core navigation hub) that maintains a stable baseline despite never being updated.")


TOP 5 FEATURES DRIVING THE MODEL:
- impressions_prev_30d: 0.270
- impressions_90d: 0.111
- days_with_impressions: 0.102
- avg_position: 0.086
- impressions_last_30d: 0.069

ERROR ANALYSIS: TOP 3 CONFIDENT FALSE POSITIVES

FP #1 (Score: 0.746)
Content: content_a5a2fbc76336
Days Stale: 103 | Impressions: 307 | CTR: 0.00% | Age: 238
Interpretation: The model heavily penalizes extreme staleness and low CTR. This page likely fits the typical 'decay' profile perfectly, but happens to be an intentionally evergreen reference page (like a homepage or core navigation hub) that maintains a stable baseline despite never being updated.

FP #2 (Score: 0.745)
Content: content_ef6e7d7cfe15
Days Stale: 104 | Impressions: 264 | CTR: 0.00% | Age: 271
Interpretation: The model heavily penalizes extreme staleness and low CTR. This page likely fits the typical 'decay' profile perfectly, but happens to be an intentionally evergreen reference page (like a homepage or core navigation hub) that maintains a sta

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.